# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 9.1 MB/s eta 0:00:00
dependencies ok


In [4]:
import re
import gc
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict, Counter
from onnx import shape_inference,helper,numpy_helper,TensorProto

In [5]:
TASK_ID = 'task170'
REVISION = "compact-v4-after-repeated-zero-score"


N = 30
K = 10


TASK_CANDIDATES = [
    Path(COMPETITION) / f"{TASK_ID}.json",
    Path.cwd() / f"{TASK_ID}.json",
    Path("/mnt/data") / f"{TASK_ID}.json",
]
TASK_PATH = next((p for p in TASK_CANDIDATES if p.exists()), None)
if TASK_PATH is None:
    raise FileNotFoundError(TASK_CANDIDATES)
with TASK_PATH.open() as f:
    task = json.load(f)

ADVERSARIAL_CASES = json.loads('[{"input":[[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,2,2,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,2,2,1,0,0,0,0,0,0,2,2,2,2,2,2,2,2,2,0,0,0,0,0,0,0,0,0],[0,3,6,2,0,0,0,0,0,0,2,2,2,2,2,2,2,2,2,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,2,2,2,2,2,2,2,2,2,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,2,2,2,0,0,0,2,2,2,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,2,2,2,0,0,0,2,2,2,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,2,2,2,0,0,0,2,2,2,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,2,2,2,2,2,2,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,2,2,2,2,2,2,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,2,2,2,2,2,2,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]],"output":[[2,2,6],[2,0,1],[0,6,2]]},{"input":[[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,2,2,2,2,2,2,2,2,2,2,2,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,2,2,2,2,2,2,2,2,2,2,2,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,2,2,2,2,2,2,2,2,2,2,2,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,2,2,2,2,2,2,2,2,2,2,2,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,2,2,2,2,2,2,2,2,2,2,2,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,2,2,2,2,2,2,2,2,2,2,2,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,2,2,2,2,2,2,2,2,2,2,2,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,2,2,2,2,2,2,2,2,2,2,2,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,2,2,2,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,2,2,2,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,2,2,2,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,2,2,2,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,2,1,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,2,3,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,4,2,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]],"output":[[2,2,1],[2,2,3],[0,0,2]]},{"input":[[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,3,3,7,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,3,3,4,0,0,0,0,0,0,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,0,0,0],[0,1,4,9,0,0,0,0,0,0,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,0,0,0],[0,0,0,0,0,0,0,0,0,0,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,0,0,0],[0,0,0,0,0,0,0,0,0,0,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,0,0,0],[0,0,0,0,0,0,0,0,0,0,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,3,3,3,3,3,3,3,3,3,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,3,3,3,3,3,3,3,3,3,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,3,3,3,3,3,3,3,3,3,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,3,3,3,3,3,3,3,3,3,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,3,3,3,3,3,3,3,3,3,0,0,0],[0,0,0,0,0,0,0,0,0,0,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,0,0,0],[0,0,0,0,0,0,0,0,0,0,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,0,0,0],[0,0,0,0,0,0,0,0,0,0,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,0,0,0],[0,0,0,0,0,0,0,0,0,0,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,0,0,0],[0,0,0,0,0,0,0,0,0,0,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]],"output":[[3,3,7],[0,3,4],[1,4,9]]},{"input":[[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,5,5,5,0,0,0,5,5,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,5,5,5,0,0,0,5,5,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,5,5,5,0,0,0,5,5,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,5,5,5,5,5,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,5,5,5,5,5,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,5,5,5,5,5,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,5,5,5,5,5,5,5,5,5,5,5,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,5,5,5,5,5,5,5,5,5,5,5,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,5,5,5,5,5,5,5,5,5,5,5,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,5,5,6,2,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,5,5,1,1,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,3,2,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,9,2,1,4,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]],"output":[[5,0,6,0],[5,5,0,0],[0,0,0,0],[9,2,1,4]]},{"input":[[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,7,7,3,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,7,7,7,7,0,0,0,0,0,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,0,0],[0,9,9,5,9,0,0,0,0,0,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,0,0],[0,7,9,6,9,0,0,0,0,0,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,0,0],[0,0,0,0,0,0,0,0,0,0,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,0,0],[0,0,0,0,0,0,0,0,0,0,7,7,7,7,0,0,0,0,7,7,7,7,7,7,7,7,0,0],[0,0,0,0,0,0,0,0,0,0,7,7,7,7,0,0,0,0,7,7,7,7,7,7,7,7,0,0],[0,0,0,0,0,0,0,0,0,0,7,7,7,7,0,0,0,0,7,7,7,7,7,7,7,7,0,0],[0,0,0,0,0,0,0,0,0,0,7,7,7,7,0,0,0,0,7,7,7,7,7,7,7,7,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,7,7,7,7,7,7,7,7,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,7,7,7,7,7,7,7,7,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,7,7,7,7,7,7,7,7,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,7,7,7,7,7,7,7,7,0,0],[0,0,0,0,0,0,0,0,0,0,7,7,7,7,0,0,0,0,0,0,0,0,7,7,7,7,0,0],[0,0,0,0,0,0,0,0,0,0,7,7,7,7,0,0,0,0,0,0,0,0,7,7,7,7,0,0],[0,0,0,0,0,0,0,0,0,0,7,7,7,7,0,0,0,0,0,0,0,0,7,7,7,7,0,0],[0,0,0,0,0,0,0,0,0,0,7,7,7,7,0,0,0,0,0,0,0,0,7,7,7,7,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]],"output":[[7,7,3,6],[7,0,7,7],[0,0,5,9],[7,0,0,9]]},{"input":[[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,6,6,6,6,6,6,6,6,6,6,6,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,6,6,6,6,6,6,6,6,6,6,6,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,6,6,6,6,6,6,6,6,6,6,6,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,6,6,6,0,0,0,6,6,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,6,6,6,0,0,0,6,6,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,6,6,6,0,0,0,6,6,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,6,6,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,6,6,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,6,6,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,6,6,6,0,0,0,6,6,6,6,6,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,6,6,6,0,0,0,6,6,6,6,6,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,6,6,6,0,0,0,6,6,6,6,6,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,6,6,4,3,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,6,6,3,7,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,8,5,3,8,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,2,6,6,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]],"output":[[6,6,4,3],[0,6,0,7],[0,0,0,8],[3,0,6,6]]}]')
OUT_DIR = Path.cwd() / f"{TASK_ID}_compact_v4"
ONNX_PATH = OUT_DIR / f"{TASK_ID}.onnx"
SUMMARY_PATH = OUT_DIR / f"{TASK_ID}_validation_summary_v4.json"
SUBMISSION_PATH = Path.cwd() / "submission.zip"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("task:", TASK_PATH)

task: /kaggle/input/competitions/neurogolf-2026/task170.json


In [6]:
def encode_grid(grid):
    arr = np.asarray(grid, dtype=np.int64)
    h, w = arr.shape
    assert h <= N and w <= N, (h, w)
    out = np.zeros((1, K, N, N), dtype=np.float32)
    one_hot = np.eye(K, dtype=np.float32)[arr]
    out[0, :, :h, :w] = one_hot.transpose(2, 0, 1)
    return out

def expected_tensor(grid):
    return encode_grid(grid)

def tensor_contract(y, grid):
    h, w = len(grid), len(grid[0])
    target = expected_tensor(grid)
    outside = y.copy()
    outside[:, :, :h, :w] = 0
    return {
        "exact": bool(np.array_equal(y, target)),
        "outside_zero": bool(np.all(outside == 0)),
        "inside_one_hot": bool(np.all(y[:, :, :h, :w].sum(axis=1) == 1.0)),
    }

def stable_hash(example):
    payload = json.dumps(example["input"], separators=(",", ":")).encode()
    return hashlib.sha256(payload).hexdigest()


In [7]:
arc_gen_all = task.get("arc-gen", [])
arc_gen_compatible = [e for e in arc_gen_all if max(len(e["input"]), len(e["input"][0]), len(e["output"]), len(e["output"][0])) <= N]
arc_gen_sorted = sorted(arc_gen_compatible, key=stable_hash)
holdout_n = int(np.ceil(0.60 * len(arc_gen_sorted)))
arc_gen_development = arc_gen_sorted[:-holdout_n] if holdout_n else arc_gen_sorted
arc_gen_holdout = arc_gen_sorted[-holdout_n:] if holdout_n else []


print("train/test/arc-gen:", len(task["train"]), len(task["test"]), len(arc_gen_all))
print("60% evaluation:", len(arc_gen_holdout), "adversarial:", len(ADVERSARIAL_CASES))


train/test/arc-gen: 3 1 262
60% evaluation: 158 adversarial: 6


In [8]:
class Base(nn.Module):
 def __init__(self):
  super().__init__()
  coord=torch.arange(N,dtype=torch.float32)
  self.register_buffer('coord',coord)
  self.register_buffer('colors',torch.arange(K,dtype=torch.float32))
  self.register_buffer('rr',coord.view(N,1).expand(N,N))
  self.register_buffer('cc',coord.view(1,N).expand(N,N))
 def canvas(self,h,w):return (self.rr[None]<h[:,None,None])&(self.cc[None]<w[:,None,None])

class Task170Compact(Base):
 def forward(self,x):
  active=x.sum(1)>0
  H=active.any(2).float().sum(1);W=active.any(1).float().sum(1)
  labels=x.argmax(1)
  counts=x.sum((2,3));bg=counts.argmax(1).float();bgoh=(self.colors.view(1,K)==bg[:,None]).float()
  # Solid 3x3 witnesses identify the scaled block lattice while ignoring
  # isolated or 2x2 repetitions of the same color in the compact palette.
  q=F.avg_pool2d(x,kernel_size=3,stride=1)==1.0
  qcnt=q.float().sum((2,3))
  qcnt=torch.where(self.colors.view(1,K)!=bg[:,None],qcnt,torch.full_like(qcnt,-1))
  sc=qcnt.argmax(1).float();scoh=(self.colors.view(1,K)==sc[:,None]).float()
  qsel=(q.float()*scoh[:,:,None,None]).sum(1)>0.5
  # q coordinates are 3x3 top-lefts; expanding the extrema by two pixels
  # recovers the exact square bounding box of the scaled lattice.
  qcoord=self.coord[:N-2]
  rowany=qsel.any(2);colany=qsel.any(1)
  bigq=torch.full_like(qcoord,1000.0);smallq=torch.full_like(qcoord,-1000.0)
  r0=torch.where(rowany,qcoord.view(1,N-2),bigq.view(1,N-2)).amin(1)
  r1=torch.where(rowany,qcoord.view(1,N-2),smallq.view(1,N-2)).amax(1)+2.0
  c0=torch.where(colany,qcoord.view(1,N-2),bigq.view(1,N-2)).amin(1)
  c1=torch.where(colany,qcoord.view(1,N-2),smallq.view(1,N-2)).amax(1)+2.0
  big=torch.full_like(self.coord,1000.0);small=torch.full_like(self.coord,-1000.0)
  shape_box=(self.rr[None]>=r0[:,None,None])&(self.rr[None]<=r1[:,None,None])&(self.cc[None]>=c0[:,None,None])&(self.cc[None]<=c1[:,None,None])
  shape=(labels==sc[:,None,None])&shape_box
  # compact palette = non-background pixels outside the large shape square
  palpix=active&(labels!=bg[:,None,None])&(~shape_box)
  prow=palpix.any(2);pcol=palpix.any(1)
  pr0=torch.where(prow,self.coord.view(1,N),big.view(1,N)).amin(1)
  pr1=torch.where(prow,self.coord.view(1,N),small.view(1,N)).amax(1)
  pc0=torch.where(pcol,self.coord.view(1,N),big.view(1,N)).amin(1)
  pc1=torch.where(pcol,self.coord.view(1,N),small.view(1,N)).amax(1)
  n=torch.maximum(pr1-pr0+1,pc1-pc0+1)
  # crop palette
  oi=self.coord.view(1,N,1);ii=self.coord.view(1,1,N)
  rmap=(oi<n[:,None,None])&(ii==pr0[:,None,None]+oi)
  cmap=(oi<n[:,None,None])&(ii==pc0[:,None,None]+oi)
  palette=torch.matmul(torch.matmul(rmap.float().unsqueeze(1),x),cmap.float().transpose(1,2).unsqueeze(1))
  # coarse occupancy from exact equal blocks
  sh=r1-r0+1;sw=c1-c0+1;k=sh/n
  rbin=(oi<n[:,None,None])&(ii>=r0[:,None,None]+oi*k[:,None,None])&(ii<r0[:,None,None]+(oi+1)*k[:,None,None])
  cbin=(oi<n[:,None,None])&(ii>=c0[:,None,None]+oi*k[:,None,None])&(ii<c0[:,None,None]+(oi+1)*k[:,None,None])
  occ=torch.matmul(torch.matmul(rbin.float().unsqueeze(1),shape[:,None].float()),cbin.float().transpose(1,2).unsqueeze(1))[:,0]>0.5
  canv=self.canvas(n,n)
  out=palette*occ[:,None].float()+bgoh[:,:,None,None]*(canv&(~occ))[:,None].float()
  return out*canv[:,None].float()


model = Task170Compact().eval()


In [9]:
def validate_eager(name, examples):
    bad = []
    with torch.no_grad():
        for i, ex in enumerate(examples):
            y = model(torch.from_numpy(encode_grid(ex["input"]))).cpu().numpy()
            if not np.array_equal(y, expected_tensor(ex["output"])):
                bad.append(i)
    print(name, len(examples)-len(bad), "/", len(examples), "bad", bad[:10])
    assert not bad, (name, bad[:10])

validate_eager("eager/train", task["train"])
validate_eager("eager/test", task["test"])
validate_eager("eager/adversarial", ADVERSARIAL_CASES)


eager/train 3 / 3 bad []
eager/test 1 / 1 bad []
eager/adversarial 6 / 6 bad []


In [10]:
dummy = torch.from_numpy(encode_grid(task["train"][0]["input"]))
with torch.no_grad():
    eager_dummy = model(dummy)
assert list(eager_dummy.shape) == [1, 10, 30, 30]
torch.onnx.export(
    model, dummy, str(ONNX_PATH),
    input_names=["input"], output_names=["output"],
    opset_version=18, do_constant_folding=True,
    dynamic_axes=None, dynamo=False, external_data=False,
)
onnx_model = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(onnx_model)
inferred = shape_inference.infer_shapes(onnx_model)
ops = Counter(node.op_type for node in onnx_model.graph.node)
FORBIDDEN = {"Loop", "Scan", "NonZero", "Unique", "Script", "Function"}
EXTENDED_AVOID = {"Einsum", "ScatterElements", "ScatterND", "GatherElements"}
forbidden = sorted(set(ops) & FORBIDDEN)
extended_present = sorted(set(ops) & EXTENDED_AVOID)
model_size = ONNX_PATH.stat().st_size
node_count = len(onnx_model.graph.node)
function_count = len(onnx_model.functions)
print("model bytes:", model_size, "nodes:", node_count)
print("operators:", dict(ops))
print("forbidden:", forbidden, "extended avoid:", extended_present)
assert model_size < 1_400_000
assert not forbidden
assert not extended_present
assert function_count == 0
assert node_count < 1000

del dummy, eager_dummy, model
gc.collect()
try:
    ctypes.CDLL("libc.so.6").malloc_trim(0)
except Exception:
    pass


/tmp/ipykernel_15/3810285047.py:5: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


model bytes: 27460 nodes: 223
operators: {'Constant': 69, 'ReduceSum': 8, 'Greater': 7, 'ArgMax': 3, 'Cast': 19, 'Unsqueeze': 32, 'Equal': 8, 'AveragePool': 1, 'Not': 4, 'Where': 9, 'Mul': 6, 'ReduceMin': 4, 'ReduceMax': 4, 'Add': 12, 'GreaterOrEqual': 4, 'LessOrEqual': 2, 'And': 14, 'Sub': 3, 'Max': 1, 'Less': 5, 'MatMul': 4, 'Transpose': 2, 'Div': 1, 'Gather': 1}
forbidden: [] extended avoid: []


In [11]:
so = ort.SessionOptions()
so.intra_op_num_threads = 1
so.inter_op_num_threads = 1
so.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
session = ort.InferenceSession(str(ONNX_PATH), sess_options=so, providers=["CPUExecutionProvider"])
input_shape = list(session.get_inputs()[0].shape)
output_shape = list(session.get_outputs()[0].shape)
assert input_shape == [1, 10, 30, 30], input_shape
assert output_shape == [1, 10, 30, 30], output_shape

def validate_onnx(name, examples):
    exact = outside = onehot = 0
    bad = []
    for i, ex in enumerate(examples):
        y = session.run(None, {"input": encode_grid(ex["input"])})[0]
        c = tensor_contract(y, ex["output"])
        exact += int(c["exact"]); outside += int(c["outside_zero"]); onehot += int(c["inside_one_hot"])
        if not c["exact"]:
            bad.append(i)
    result = {"ok": exact, "total": len(examples), "outside_zero_ok": outside, "inside_one_hot_ok": onehot, "bad_first10": bad[:10]}
    print(name, result)
    assert exact == len(examples), (name, bad[:10])
    assert outside == len(examples)
    assert onehot == len(examples)
    return result

# Runtime is part of this revision: the failed v3 graphs were much more expensive.
bench_x = encode_grid(task["test"][0]["input"])
for _ in range(10):
    session.run(None, {"input": bench_x})
t0 = time.perf_counter()
for _ in range(100):
    session.run(None, {"input": bench_x})
avg_ms = (time.perf_counter() - t0) * 10.0
print("average ONNX latency ms:", avg_ms)
assert avg_ms < 100.0

summary = {
    "task_id": TASK_ID,
    "revision": REVISION,
    "task_type": 'nonlocal scaled binary mask applied to a compact palette',
    "structural_rule": 'Use monochrome 3x3 witnesses to identify the large block lattice, recover its exact square bounds, locate the separate compact palette, downsample the lattice into its coarse occupancy mask, and retain palette entries only where that mask is active.',
    "modelling_change": 'The v3 graph searched many lattice decompositions and exceeded ten thousand nodes. This version isolates the scale structurally with one AveragePool and four MatMul operations.',
    "input_shape": input_shape,
    "output_shape": output_shape,
    "onnx_size_bytes": model_size,
    "onnx_node_count": node_count,
    "average_latency_ms": avg_ms,
    "ops": dict(ops),
    "forbidden_ops": forbidden,
    "extended_avoid_ops": extended_present,
    "function_count": function_count,
    "train": validate_onnx("onnx/train", task["train"]),
    "test": validate_onnx("onnx/test", task["test"]),
    "arc_gen_holdout_60pct": validate_onnx("onnx/arc-gen 60%", arc_gen_holdout),
    "arc_gen_all_diagnostic": validate_onnx("onnx/arc-gen all", arc_gen_compatible),
    "adversarial": validate_onnx("onnx/adversarial", ADVERSARIAL_CASES),
}
with SUMMARY_PATH.open("w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))


average ONNX latency ms: 0.30187017000002925
onnx/train {'ok': 3, 'total': 3, 'outside_zero_ok': 3, 'inside_one_hot_ok': 3, 'bad_first10': []}
onnx/test {'ok': 1, 'total': 1, 'outside_zero_ok': 1, 'inside_one_hot_ok': 1, 'bad_first10': []}
onnx/arc-gen 60% {'ok': 158, 'total': 158, 'outside_zero_ok': 158, 'inside_one_hot_ok': 158, 'bad_first10': []}
onnx/arc-gen all {'ok': 262, 'total': 262, 'outside_zero_ok': 262, 'inside_one_hot_ok': 262, 'bad_first10': []}
onnx/adversarial {'ok': 6, 'total': 6, 'outside_zero_ok': 6, 'inside_one_hot_ok': 6, 'bad_first10': []}
{
  "task_id": "task170",
  "revision": "compact-v4-after-repeated-zero-score",
  "task_type": "nonlocal scaled binary mask applied to a compact palette",
  "structural_rule": "Use monochrome 3x3 witnesses to identify the large block lattice, recover its exact square bounds, locate the separate compact palette, downsample the lattice into its coarse occupancy mask, and retain palette entries only where that mask is active.",
  "

In [12]:
if SUBMISSION_PATH.exists():
    SUBMISSION_PATH.unlink()
with zipfile.ZipFile(SUBMISSION_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(ONNX_PATH, arcname=f"{TASK_ID}.onnx")
with zipfile.ZipFile(SUBMISSION_PATH) as zf:
    assert zf.namelist() == [f"{TASK_ID}.onnx"]
    info = zf.infolist()[0]
    print("submission.zip:", info.filename, info.file_size, "bytes")


submission.zip: task170.onnx 27460 bytes
